# BeatSaberMaker: trening na Google Colab

Kroki:
1. Wgraj dwa pliki do `Mój dysk/BeatSaberMaker/`: `bsmaker-train-data.tar` (dane) i `bsmaker-code.zip` (kod, z `scripts/make_colab_bundle.py`).
2. Runtime -> Change runtime type -> GPU.
3. Runtime -> Run all.

Po rozłączeniu sesji: "Run all" jeszcze raz. Dane zostaną rozpakowane ponownie, a trening wznowi się z ostatniego checkpointu na Dysku.

Uwagi:
- Karta przeglądarki z notebookiem musi zostać otwarta (może to być dowolna maszyna działająca non-stop, np. Raspberry Pi z Chromium).
- Plan Dysku Google musi pomieścić plik 40 GB.

## 1. Konfiguracja (tu zmieniasz parametry)

In [ ]:
# --- CONFIG: edit only this cell ---
MAX_STEPS_TIMING = 40000
MAX_STEPS_PATTERN = 60000
BATCH_SIZE = 32
NUM_WORKERS = 4

## 2. Sprawdzenie GPU

In [ ]:
import subprocess
import torch

import shutil
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
if not torch.cuda.is_available():
    raise SystemExit("No GPU: use Runtime -> Change runtime type -> GPU, then run all again.")
print("CUDA OK:", torch.cuda.get_device_name(0))

## 3. Podłączenie Dysku Google

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/BeatSaberMaker')
TAR = DRIVE / 'bsmaker-train-data.tar'
ZIP = DRIVE / 'bsmaker-code.zip'
missing = [str(p) for p in (TAR, ZIP) if not p.is_file()]
if missing:
    raise SystemExit("Missing files on Drive:\n  " + "\n  ".join(missing))
print("Found:", TAR.name, ZIP.name)

## 4. Kod: rozpakowanie i zależności

In [ ]:
import os, shutil, subprocess, sys, zipfile

CODE = Path('/content/bsmaker')
if CODE.exists():
    shutil.rmtree(CODE)
with zipfile.ZipFile(ZIP) as zf:
    zf.extractall(CODE)
print("Bundle commit:", (CODE / 'BUNDLE_COMMIT.txt').read_text().strip())

# Colab's Python is older than pyproject requires, so the package is not pip-installed;
# only the missing runtime dependencies are (torch and numpy are preinstalled).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'librosa', 'soundfile', 'typer', 'httpx'], check=True)

(CODE / 'bsmaker.toml').write_text("data_dir = '/content/data'\ncustom_levels = '/content/none'\n")

ENV = dict(os.environ, PYTHONPATH=str(CODE / 'src'), PYTHONUNBUFFERED='1')

def run(args):
    """Run `python -m bsmaker ...` from the code folder, printing its output line by line."""
    cmd = [sys.executable, '-u', '-m', 'bsmaker', *map(str, args)]
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=CODE, env=ENV, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
        code = proc.wait()
    finally:
        if proc.poll() is None:
            proc.kill()
            proc.wait()
    if code != 0:
        raise RuntimeError(f"bsmaker exited with code {code}")

## 5. Dane: rozpakowanie z Dysku (powtarzalne)

In [ ]:
import time

DATA = Path('/content/data')
PART = Path('/content/data.partial')
DONE = DATA / '.done'
NEED_GB = 42

def count_npy(root):
    return sum(1 for _ in (root / 'features').glob('*.npy'))

def check_data(root):
    n = count_npy(root)
    print("feature files:", n)
    assert n > 14000, f"too few .npy files ({n}): unpacking incomplete?"
    for name in ('events.npz', 'index.json'):
        assert (root / 'events' / name).is_file(), f"missing events/{name}"

def unpack_data():
    if DONE.is_file():
        print("Data already unpacked (.done marker), skipping.")
        return
    for d in (DATA, PART):  # leftovers of an interrupted unpack
        shutil.rmtree(d, ignore_errors=True)
    free_gb = shutil.disk_usage('/content').free / 1e9
    print(f"Free disk: {free_gb:.1f} GB (need >= {NEED_GB} GB)")
    if free_gb < NEED_GB:
        raise SystemExit(f"Not enough disk: {free_gb:.1f} GB free, need >= {NEED_GB} GB.")
    PART.mkdir(parents=True)
    proc = subprocess.Popen(['tar', '-xf', str(TAR), '-C', str(PART)])
    start = time.time()
    try:
        while proc.poll() is None:
            time.sleep(30)
            used = shutil.disk_usage('/content').used / 1e9
            print(f"  {time.time() - start:6.0f}s  disk used {used:.1f} GB", flush=True)
    finally:
        if proc.poll() is None:
            proc.kill()
            proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f"tar failed with exit code {proc.returncode}")
    check_data(PART)
    PART.rename(DATA)
    DONE.write_text('ok')

unpack_data()
check_data(DATA)
print("Data OK")

## 6. Szybki test (20 kroków)

In [ ]:
PROBE = Path('/content/probe_timing')
run(['train', 'timing', '--max-steps', 20, '--eval-every', 20, '--save-every', 20,
     '--out', PROBE, '--num-workers', 2])
shutil.rmtree(PROBE, ignore_errors=True)

## 7. Trening modelu timing (po rozłączeniu uruchom ponownie: wznawia)

In [ ]:
run(['train', 'timing', '--out', DRIVE / 'checkpoints' / 'timing', '--max-steps', MAX_STEPS_TIMING,
     '--batch-size', BATCH_SIZE, '--num-workers', NUM_WORKERS])

## 8. Trening modelu pattern (też wznawia)

In [ ]:
run(['train', 'pattern', '--out', DRIVE / 'checkpoints' / 'pattern', '--max-steps', MAX_STEPS_PATTERN,
     '--batch-size', BATCH_SIZE, '--num-workers', NUM_WORKERS])

## 9. Podsumowanie

Pobierz z Dysku i zapisz lokalnie:
- `BeatSaberMaker/checkpoints/timing/best.pt` -> `F:\BeatSaberMakerData\checkpoints\timing\best.pt`
- `BeatSaberMaker/checkpoints/pattern/best.pt` -> `F:\BeatSaberMakerData\checkpoints\pattern\best.pt`

In [ ]:
for name in ('timing', 'pattern'):
    folder = DRIVE / 'checkpoints' / name
    print(f"=== {name} ===")
    log = folder / 'log.jsonl'
    if log.is_file():
        print(''.join(log.read_text().splitlines(True)[-5:]))
    else:
        print("no log.jsonl")
    best = folder / 'best.pt'
    if best.is_file():
        ck = torch.load(best, weights_only=True, map_location='cpu')
        print("best:", ck.get('best'))
    else:
        print("no best.pt")

## 10. Zapis Dysku

Uruchom tę komórkę przed pobraniem checkpointów lub zamknięciem runtime: wymusza zapis zbuforowanych plików na Dysk.

In [ ]:
drive.flush_and_unmount()
print("Drive flushed.")